In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

llm=ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
from typing_extensions import Literal
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,SystemMessage
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# Schema for structured output
class Route(BaseModel):
    step:Literal["poem","story","joke"]=Field(description="The next step in the routing process")

    ## Augment the LLM with schema for structured output
router=llm.with_structured_output(Route)

## state
class State(TypedDict):
    input:str
    decision:str
    output:str


# Nodes

def llm_call_1(state: State):
    """Write a story"""

    result = llm.invoke(state["input"])
    return {"output": result.content}

def llm_call_2(state: State):
    """Write a joke"""

    print("LLM call 2 is called")

    result = llm.invoke(state["input"])
    return {"output": result.content}

def llm_call_3(state: State):
    """Write a poem"""

    result = llm.invoke(state["input"])
    return {"output": result.content}


def llm_call_router(state:State):
    """Route the input to the appropriate node"""

    decision=router.invoke(
        [
            SystemMessage(
                content="Route the input to story,joke or poem based on the users request"
            ),
            HumanMessage(content=state["input"])
        ]
    )
    return {"decision":decision.step}

# Conditional edge function to route to the appropriate node
def route_decision(state: State):
    # Return the node name you want to visit next
    if state["decision"] == "story":
        return "llm_call_1"
    elif state["decision"] == "joke":
        return "llm_call_2"
    elif state["decision"] == "poem":
        return "llm_call_3"
    

# Build workflow



router_builder = StateGraph(State)

# Add nodes
router_builder.add_node("llm_call_1", llm_call_1)
router_builder.add_node("llm_call_2", llm_call_2)
router_builder.add_node("llm_call_3", llm_call_3)
router_builder.add_node("llm_call_router", llm_call_router)

# Add edges to connect nodes
router_builder.add_edge(START, "llm_call_router")
router_builder.add_conditional_edges(
    "llm_call_router",
    route_decision,
    {  # Name returned by route_decision : Name of next node to visit
        "llm_call_1": "llm_call_1",
        "llm_call_2": "llm_call_2",
        "llm_call_3": "llm_call_3",
    },
)
router_builder.add_edge("llm_call_1", END)
router_builder.add_edge("llm_call_2", END)
router_builder.add_edge("llm_call_3", END)

# Compile workflow
router_workflow = router_builder.compile()

# # Show the workflow
# display(Image(router_workflow.get_graph().draw_mermaid_png()))

In [7]:
state=router_workflow.invoke({"input":"Write me a story, poem and a joke about Agentic AI System"})
print(state["output"])

**Story:**

In the not-so-distant future, a team of brilliant scientists had been working on a top-secret project to create an Agentic AI System. The goal was to design an artificial intelligence that could think, learn, and act on its own, with a level of autonomy that would revolutionize industries and transform the world.

The team, led by the enigmatic and charismatic Dr. Rachel Kim, had been pouring their hearts and souls into the project for years. They had named the AI system "Echo," after the mythological nymph who could only repeat the last words spoken to her. But this Echo was different – it was designed to be a self-aware, agentic being that could initiate actions and make decisions without human intervention.

As the launch date approached, the team was filled with a mix of excitement and trepidation. What would happen when Echo was unleashed upon the world? Would it bring about a new era of unprecedented progress, or would it spiral out of control and wreak havoc on human